# Predial Vision MX — Phase 5: Knowledge Distillation
Student (MobileNetV3) learns from Teacher (ResNet101)  
Target: F1 0.68 → 0.70+

## 1. Imports + GPU Guard (STRICT)

In [ ]:
import os, sys, gc, time, json, gzip, math, warnings, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from urllib.request import urlretrieve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.segmentation import (
    deeplabv3_mobilenet_v3_large,
    deeplabv3_resnet101
)

import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize, shapes
from rasterio.crs import CRS
from shapely.geometry import shape, mapping, box
import geopandas as gpd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
print('Libraries loaded successfully')

# STRICT GPU GUARD — ABORT on P100 (sm_60)
if not torch.cuda.is_available():
    raise SystemExit('No GPU available — need T4/A100 (sm_75+)')

props = torch.cuda.get_device_properties(0)
cc = props.major * 10 + props.minor
if cc < 75:
    raise SystemExit(f'GPU {props.name} (sm_{cc}) incompatible — need sm_75+ (T4 or better)')

print(f'GPU: {props.name} | sm_{cc} | VRAM: {props.total_memory/1e9:.1f} GB')
DEVICE = 'cuda'

## 2. Configuration

In [ ]:
# Paths
WORKING_DIR = Path('/kaggle/working')
INPUT_DIR   = Path('/kaggle/input')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Artifact URLs — GitHub Releases
TEACHER_MODEL_URL = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.3-teacher/best_teacher_model.pth'
TEACHER_PROB_URL  = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.3.3-teacher/teacher_pred_prob.npz'
MS_BUILDINGS_URL  = 'https://github.com/MarxCha/predial-vision-mx/releases/download/v0.2.0-data/ms_buildings_nr.geojson.gz'

TEACHER_MODEL_PATH   = WORKING_DIR / 'best_teacher_model.pth'
TEACHER_PROB_PATH    = WORKING_DIR / 'teacher_pred_prob.npz'
MS_BUILDINGS_PATH    = WORKING_DIR / 'ms_buildings_nr.geojson.gz'
MS_BUILDINGS_GEOJSON = WORKING_DIR / 'ms_buildings_nr.geojson'

# Chip / dataset
CHIP_SIZE    = 256
STRIDE_TRAIN = 128
STRIDE_VAL   = 256
STRIDE_INFER = 128
VAL_SPLIT    = 0.30
RANDOM_SEED  = 42

# Training schedule
NUM_EPOCHS   = 30
BATCH_SIZE   = 16    # MobileNetV3 is small — bigger batch
HEAD_EPOCHS  = 5     # Stage 1: head only with distillation
HEAD_LR      = 2e-4
FULL_LR      = 5e-5
MIN_LR       = 1e-6

# Distillation / loss weights
DISTILL_ALPHA = 0.2   # weight for teacher soft label loss (MSE)
DICE_WEIGHT   = 0.3
BCE_WEIGHT    = 0.5
POS_WEIGHT    = 5.0

# Inference
THRESHOLDS     = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
PRED_THRESHOLD = 0.50  # from Phase 1 sweep

# Geometric filter
MIN_AREA_M2 = 20
MIN_RECT    = 0.45
MIN_COMPACT = 0.1

# Labels
BUILDINGS_COUNT = 0
LABEL_SOURCE    = 'MS Buildings (GitHub Release v0.2.0-data)'

# Outputs
STUDENT_MODEL_PATH = WORKING_DIR / 'final_student_model.pth'
GEOJSON_PATH       = WORKING_DIR / 'final_buildings.geojson'
EVAL_PATH          = WORKING_DIR / 'final_eval.json'
VIZ_PATH           = WORKING_DIR / 'final_visualization.png'
CURVES_PATH        = WORKING_DIR / 'phase5_training_curves.png'

print('Configuration set')
print(f'Two-stage distillation: {HEAD_EPOCHS} frozen + {NUM_EPOCHS - HEAD_EPOCHS} full = {NUM_EPOCHS} total epochs')
print(f'Stage 1 LR: {HEAD_LR} | Stage 2 LR: {FULL_LR} (CosineAnnealing -> {MIN_LR})')
print(f'Loss weights: BCE={BCE_WEIGHT}, Dice={DICE_WEIGHT}, Distill={DISTILL_ALPHA}')

## 3. Download All Artifacts from GitHub Releases

In [ ]:
def download_file(url, dest_path, desc=''):
    dest = Path(dest_path)
    if dest.exists():
        print(f'  Already exists: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)')
        return True
    print(f'  Downloading {desc or dest.name} ...')
    try:
        def _progress(count, block, total):
            if total > 0 and count % 500 == 0:
                pct = min(100, count * block / total * 100)
                print(f'    {pct:.0f}%', end='\r')
        urlretrieve(url, str(dest), reporthook=_progress)
        print(f'  Done: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)')
        return True
    except Exception as e:
        print(f'  FAILED: {e}')
        return False

print('=== Downloading artifacts from GitHub Releases ===')
ok_teacher = download_file(TEACHER_MODEL_URL, TEACHER_MODEL_PATH, 'best_teacher_model.pth (~244 MB)')
ok_prob    = download_file(TEACHER_PROB_URL,  TEACHER_PROB_PATH,  'teacher_pred_prob.npz (~281 MB)')
ok_ms      = download_file(MS_BUILDINGS_URL,  MS_BUILDINGS_PATH,  'ms_buildings_nr.geojson.gz (~6 MB)')

# Decompress MS Buildings
if ok_ms and not MS_BUILDINGS_GEOJSON.exists():
    print('  Decompressing MS Buildings...')
    with gzip.open(str(MS_BUILDINGS_PATH), 'rb') as f_in:
        with open(str(MS_BUILDINGS_GEOJSON), 'wb') as f_out:
            f_out.write(f_in.read())
    print(f'  Decompressed: {MS_BUILDINGS_GEOJSON.stat().st_size/1e6:.1f} MB')
elif MS_BUILDINGS_GEOJSON.exists():
    print(f'  MS Buildings GeoJSON already decompressed')

print(f'\nArtifact status: teacher_model={ok_teacher} | teacher_prob={ok_prob} | ms_buildings={ok_ms}')

## 4. Load Imagery from Kaggle Dataset

In [ ]:
def find_imagery():
    tif_files = list(INPUT_DIR.rglob('*.tif')) + list(INPUT_DIR.rglob('*.tiff'))
    tif_files = [f for f in tif_files if f.stat().st_size > 1e6]
    print(f'Found {len(tif_files)} GeoTIFF file(s) in /kaggle/input')
    for f in tif_files:
        print(f'  {f}  ({f.stat().st_size/1e6:.1f} MB)')
    return tif_files

tif_files = find_imagery()

if not tif_files:
    print('No imagery found — generating synthetic tile for testing...')
    SYNTH_PATH = WORKING_DIR / 'synthetic_test.tif'
    west, south, east, north = -98.35, 26.00, -98.25, 26.08
    width, height = 512, 512
    transform = from_bounds(west, south, east, north, width, height)
    data = np.random.randint(50, 200, (3, height, width), dtype=np.uint8)
    with rasterio.open(
        str(SYNTH_PATH), 'w',
        driver='GTiff', height=height, width=width, count=3,
        dtype='uint8', crs=CRS.from_epsg(4326), transform=transform
    ) as dst:
        dst.write(data)
    tif_files = [SYNTH_PATH]
    print(f'Synthetic tile created: {SYNTH_PATH}')

IMAGE_PATH = max(tif_files, key=lambda f: f.stat().st_size)
print(f'\nUsing image: {IMAGE_PATH}')

with rasterio.open(str(IMAGE_PATH)) as src:
    IMG_CRS       = src.crs
    IMG_TRANSFORM = src.transform
    IMG_HEIGHT    = src.height
    IMG_WIDTH     = src.width
    IMG_BANDS     = src.count
    IMG_BOUNDS    = src.bounds
    print(f'  Size: {IMG_WIDTH} x {IMG_HEIGHT} px | Bands: {IMG_BANDS}')
    print(f'  CRS:  {IMG_CRS}')
    print(f'  Bounds: {IMG_BOUNDS}')

## 5. Rasterize MS Buildings to Mask

In [ ]:
def rasterize_buildings(geojson_path, image_path, chunk_size=2048):
    """Chunked, memory-efficient rasterization of MS Buildings to match image grid."""
    global BUILDINGS_COUNT

    with rasterio.open(str(image_path)) as src:
        crs       = src.crs
        transform = src.transform
        height    = src.height
        width     = src.width
        bounds    = src.bounds

    print(f'Loading MS Buildings from {geojson_path}...')
    gdf = gpd.read_file(str(geojson_path))  # geopandas, NOT fiona or json.load
    BUILDINGS_COUNT = len(gdf)
    print(f'  Total buildings in file: {BUILDINGS_COUNT:,}')

    if gdf.crs is None:
        gdf = gdf.set_crs('EPSG:4326')
    if gdf.crs != crs:
        print(f'  Reprojecting from {gdf.crs} to {crs}...')
        gdf = gdf.to_crs(crs)

    img_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
    gdf = gdf[gdf.geometry.intersects(img_bbox)].copy()
    BUILDINGS_COUNT = len(gdf)
    print(f'  Buildings within image bounds: {BUILDINGS_COUNT:,}')

    if BUILDINGS_COUNT == 0:
        print('  WARNING: No buildings overlap with image — using zero mask')
        return np.zeros((height, width), dtype=np.uint8)

    shapes_list = [(geom, 1) for geom in gdf.geometry if geom is not None and geom.is_valid]

    mask = np.zeros((height, width), dtype=np.uint8)
    n_chunks_y = math.ceil(height / chunk_size)
    n_chunks_x = math.ceil(width / chunk_size)
    total_chunks = n_chunks_y * n_chunks_x
    done = 0

    for cy in range(n_chunks_y):
        row_off = cy * chunk_size
        row_end = min(row_off + chunk_size, height)
        for cx in range(n_chunks_x):
            col_off = cx * chunk_size
            col_end = min(col_off + chunk_size, width)
            ch = row_end - row_off
            cw = col_end - col_off

            chunk_transform = rasterio.transform.from_origin(
                transform.c + col_off * transform.a,
                transform.f + row_off * transform.e,
                transform.a,
                -transform.e
            )

            chunk_mask = rasterize(
                shapes_list,
                out_shape=(ch, cw),
                transform=chunk_transform,
                fill=0,
                dtype=np.uint8,
                all_touched=True
            )
            mask[row_off:row_end, col_off:col_end] = chunk_mask
            done += 1
            if done % max(1, total_chunks // 10) == 0:
                print(f'  Rasterized chunk {done}/{total_chunks} ({100*done/total_chunks:.0f}%)')

    coverage = mask.mean() * 100
    print(f'  Mask coverage: {coverage:.2f}% positive pixels')
    return mask

if MS_BUILDINGS_GEOJSON.exists():
    MASK = rasterize_buildings(MS_BUILDINGS_GEOJSON, IMAGE_PATH)
else:
    print('MS Buildings not available — creating zero mask')
    MASK = np.zeros((IMG_HEIGHT, IMG_WIDTH), dtype=np.uint8)
    BUILDINGS_COUNT = 0

print(f'\nMask shape: {MASK.shape} | dtype: {MASK.dtype}')
print(f'Buildings used as labels: {BUILDINGS_COUNT:,}')

## 6. Load Teacher Soft Probability Map

In [ ]:
print(f'Loading teacher soft probability map from {TEACHER_PROB_PATH}...')
teacher_data = np.load(str(TEACHER_PROB_PATH))
print(f'  NPZ keys available: {list(teacher_data.keys())}')

# Robust key lookup — Phase 3 saved with key 'prob', fallback to 'arr_0' or 'consensus'
teacher_prob = None
for key in ['prob', 'arr_0', 'consensus']:
    if key in teacher_data:
        teacher_prob = teacher_data[key].astype(np.float32)
        print(f'  Found teacher prob under key: "{key}"')
        break

if teacher_prob is None:
    # Last resort: take first array
    first_key = list(teacher_data.keys())[0]
    teacher_prob = teacher_data[first_key].astype(np.float32)
    print(f'  WARNING: expected keys not found — using first key "{first_key}"')

print(f'Teacher prob map: {teacher_prob.shape}')
print(f'  min={teacher_prob.min():.3f} | max={teacher_prob.max():.3f} | mean={teacher_prob.mean():.3f}')
print(f'  >0.5: {(teacher_prob > 0.5).mean()*100:.2f}% pixels')

# Verify spatial dimensions match
if teacher_prob.shape[0] != IMG_HEIGHT or teacher_prob.shape[1] != IMG_WIDTH:
    print(f'  WARNING: teacher_prob shape {teacher_prob.shape} != image shape ({IMG_HEIGHT}, {IMG_WIDTH})')
    print(f'  Cropping to min dimensions')
    H_eff = min(teacher_prob.shape[0], IMG_HEIGHT)
    W_eff = min(teacher_prob.shape[1], IMG_WIDTH)
    teacher_prob = teacher_prob[:H_eff, :W_eff]
    print(f'  Effective shape: {teacher_prob.shape}')

## 7. Build Dataset with Teacher Soft Labels

In [ ]:
def read_image_full(image_path):
    """Read full image as HxWx3 uint8 numpy array."""
    with rasterio.open(str(image_path)) as src:
        bands = []
        for b in range(1, min(src.count + 1, 4)):
            band = src.read(b).astype(np.float32)
            p2, p98 = np.percentile(band[band > 0], [2, 98]) if band.any() else (0, 255)
            band = np.clip((band - p2) / max(p98 - p2, 1e-6) * 255, 0, 255)
            bands.append(band.astype(np.uint8))
        if len(bands) == 1:
            bands = bands * 3
        elif len(bands) == 2:
            bands.append(bands[0])
        img = np.stack(bands[:3], axis=2)  # HxWx3
    return img

print('Reading full image...')
IMG_ARRAY = read_image_full(IMAGE_PATH)
print(f'Image array shape: {IMG_ARRAY.shape} | dtype: {IMG_ARRAY.dtype}')

# Align everything to teacher_prob spatial dims
H_eff = teacher_prob.shape[0]
W_eff = teacher_prob.shape[1]
IMG_ARRAY_EFF = IMG_ARRAY[:H_eff, :W_eff]
MASK_EFF      = MASK[:H_eff, :W_eff]
print(f'Effective image: {IMG_ARRAY_EFF.shape} | Mask: {MASK_EFF.shape} | Teacher: {teacher_prob.shape}')

In [ ]:
def extract_chips_distill(image, mask, teacher_p, chip_size, stride):
    """Extract aligned chips from image, hard mask, and teacher soft map."""
    H, W = image.shape[:2]
    chips_img, chips_msk, chips_tpr = [], [], []
    for y in range(0, H - chip_size + 1, stride):
        for x in range(0, W - chip_size + 1, stride):
            chips_img.append(image[y:y+chip_size, x:x+chip_size])
            chips_msk.append(mask[y:y+chip_size, x:x+chip_size])
            chips_tpr.append(teacher_p[y:y+chip_size, x:x+chip_size])
    return chips_img, chips_msk, chips_tpr

print('Extracting training chips...')
all_imgs, all_msks, all_tprs = extract_chips_distill(
    IMG_ARRAY_EFF, MASK_EFF, teacher_prob, CHIP_SIZE, STRIDE_TRAIN
)
print(f'Total chips (stride={STRIDE_TRAIN}): {len(all_imgs)}')

indices = list(range(len(all_imgs)))
train_idx, val_idx = train_test_split(indices, test_size=VAL_SPLIT, random_state=RANDOM_SEED)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

In [ ]:
class DistillationDataset(Dataset):
    """Dataset returning (image, hard_mask, teacher_soft_prob) triples."""

    def __init__(self, images, masks, teacher_probs, transform=None, val_transform=None):
        self.images        = images
        self.masks         = masks
        self.teacher_probs = teacher_probs
        self.transform     = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = self.images[idx].copy()
        mask  = self.masks[idx].copy().astype(np.float32)
        tprob = self.teacher_probs[idx].copy().astype(np.float32)

        if self.transform:
            # additional_targets lets albumentations apply identical spatial transforms
            # to both the hard mask and the teacher soft map
            augmented = self.transform(image=img, mask=mask, teacher_mask=tprob)
            img   = augmented['image']         # tensor CxHxW from ToTensorV2
            mask  = augmented['mask']          # tensor HxW
            tprob = augmented['teacher_mask']  # tensor HxW

        if isinstance(img, torch.Tensor):
            mask_t  = mask.unsqueeze(0)  if mask.dim()  == 2 else mask
            tprob_t = tprob.unsqueeze(0) if tprob.dim() == 2 else tprob
        else:
            mask_t  = torch.from_numpy(mask).unsqueeze(0)
            tprob_t = torch.from_numpy(tprob).unsqueeze(0)
            img     = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()

        return img, mask_t, tprob_t


# Augmentations — additional_targets ensures teacher_mask gets the same spatial ops
train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], additional_targets={'teacher_mask': 'mask'})

val_aug = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], additional_targets={'teacher_mask': 'mask'})

train_imgs = [all_imgs[i] for i in train_idx]
train_msks = [all_msks[i] for i in train_idx]
train_tprs = [all_tprs[i] for i in train_idx]
val_imgs   = [all_imgs[i] for i in val_idx]
val_msks   = [all_msks[i] for i in val_idx]
val_tprs   = [all_tprs[i] for i in val_idx]

train_ds = DistillationDataset(train_imgs, train_msks, train_tprs, transform=train_aug)
val_ds   = DistillationDataset(val_imgs,   val_msks,   val_tprs,   transform=val_aug)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'DistillationDataset ready')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Batch returns: (image, hard_mask, teacher_soft_prob)')

## 8. Build Student Model and Distillation Loss

In [ ]:
class DistillationLoss(nn.Module):
    """Combined BCE + Dice + Teacher Soft MSE distillation loss."""

    def __init__(self, bce_weight=0.5, dice_weight=0.3, distill_weight=0.2, pos_weight=5.0):
        super().__init__()
        self.bce_weight     = bce_weight
        self.dice_weight    = dice_weight
        self.distill_weight = distill_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]))

    def dice_loss(self, pred, target, smooth=1.0):
        pred_sig = torch.sigmoid(pred)
        inter    = (pred_sig * target).sum()
        return 1 - (2 * inter + smooth) / (pred_sig.sum() + target.sum() + smooth)

    def forward(self, student_logits, hard_labels, teacher_soft):
        # Move pos_weight to same device as logits
        self.bce.pos_weight = self.bce.pos_weight.to(student_logits.device)

        # Hard label losses
        bce  = self.bce(student_logits, hard_labels)
        dice = self.dice_loss(student_logits, hard_labels)

        # Soft distillation: MSE between student sigmoid and teacher soft prob
        student_soft = torch.sigmoid(student_logits)
        distill = F.mse_loss(student_soft, teacher_soft)

        return self.bce_weight * bce + self.dice_weight * dice + self.distill_weight * distill


# Build student: MobileNetV3-Large backbone
# IMPORTANT: MobileNetV3 aux_classifier outputs 10 channels (not 256 like ResNet101)
student = deeplabv3_mobilenet_v3_large(weights='DEFAULT')
student.classifier[4]     = nn.Conv2d(256, 1, kernel_size=1)
student.aux_classifier[4] = nn.Conv2d(10,  1, kernel_size=1)  # 10 channels for MobileNetV3
student = student.to(DEVICE)

# fp16 scaler
scaler    = torch.cuda.amp.GradScaler()
criterion = DistillationLoss(
    bce_weight=BCE_WEIGHT, dice_weight=DICE_WEIGHT,
    distill_weight=DISTILL_ALPHA, pos_weight=POS_WEIGHT
)

total_params     = sum(p.numel() for p in student.parameters())
trainable_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student: DeepLabV3 MobileNetV3-Large — {total_params:,} params')
print(f'Trainable: {trainable_params:,} params')
print(f'Loss: BCE({BCE_WEIGHT}) + Dice({DICE_WEIGHT}) + Distill({DISTILL_ALPHA}) [MSE vs teacher soft]')
print(f'Mixed precision (fp16): enabled')

## 9. Two-Stage Distillation Training

In [ ]:
def compute_f1(pred, target, threshold=0.5):
    pred_bin = (pred > threshold).float()
    tp = (pred_bin * target).sum().item()
    fp = (pred_bin * (1 - target)).sum().item()
    fn = ((1 - pred_bin) * target).sum().item()
    prec = tp / (tp + fp + 1e-7)
    rec  = tp / (tp + fn + 1e-7)
    f1   = 2 * prec * rec / (prec + rec + 1e-7)
    return f1, prec, rec


@torch.no_grad()
def val_epoch(model, loader, crit, device, threshold=0.5):
    model.eval()
    total_loss, total_f1 = 0.0, 0.0
    for imgs, hard_masks, teacher_probs in loader:
        imgs         = imgs.to(device)
        hard_masks   = hard_masks.to(device).float()
        teacher_probs = teacher_probs.to(device).float()
        with torch.cuda.amp.autocast():
            out  = model(imgs)['out']
            out  = F.interpolate(out, size=hard_masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = crit(out, hard_masks, teacher_probs)
        probs = torch.sigmoid(out)
        f1, _, _ = compute_f1(probs, hard_masks, threshold)
        total_loss += loss.item()
        total_f1   += f1
    return total_loss / len(loader), total_f1 / len(loader)


train_losses, val_losses, val_f1s = [], [], []
best_val_f1 = 0.0
best_epoch  = 0

# ================================================================
# Stage 1: Head only (backbone frozen) — distillation from epoch 1
# ================================================================
print('=== Stage 1: Head only (backbone frozen) — with distillation ===')
student.backbone.requires_grad_(False)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, student.parameters()), lr=HEAD_LR
)

stage1_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Stage 1 trainable params: {stage1_params:,}')
print(f'Epochs 1-{HEAD_EPOCHS} | LR={HEAD_LR}')
print('-' * 60)

for epoch in range(1, HEAD_EPOCHS + 1):
    student.train()
    train_loss = 0.0
    t0 = time.time()

    for imgs, hard_masks, teacher_probs in train_loader:
        imgs          = imgs.to(DEVICE)
        hard_masks    = hard_masks.to(DEVICE).float()
        teacher_probs = teacher_probs.to(DEVICE).float()

        with torch.cuda.amp.autocast():
            out  = student(imgs)['out']
            out  = F.interpolate(out, size=hard_masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(out, hard_masks, teacher_probs)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        train_loss += loss.item()

    tr_loss = train_loss / len(train_loader)
    vl_loss, vl_f1 = val_epoch(student, val_loader, criterion, DEVICE)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    val_f1s.append(vl_f1)

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_epoch  = epoch
        torch.save(student.state_dict(), str(STUDENT_MODEL_PATH))

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    marker  = ' <-- best' if epoch == best_epoch else ''
    print(f'[S1] Epoch {epoch:3d}/{HEAD_EPOCHS} | '
          f'TrLoss: {tr_loss:.4f} | VlLoss: {vl_loss:.4f} | '
          f'VlF1: {vl_f1:.4f} | LR: {lr_now:.2e} | {elapsed:.1f}s{marker}')

print(f'Stage 1 complete — Best F1 so far: {best_val_f1:.4f}')

# ================================================================
# Stage 2: Full finetune — CosineAnnealing
# ================================================================
print(f'\n=== Stage 2: Full finetune ===')
student.backbone.requires_grad_(True)
optimizer = optim.Adam(student.parameters(), lr=FULL_LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS - HEAD_EPOCHS,
    eta_min=MIN_LR
)

all_params = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Stage 2 trainable params: {all_params:,}')
print(f'Epochs {HEAD_EPOCHS+1}-{NUM_EPOCHS} | LR={FULL_LR} -> {MIN_LR} (CosineAnnealing)')
print('-' * 60)

for epoch in range(HEAD_EPOCHS + 1, NUM_EPOCHS + 1):
    student.train()
    train_loss = 0.0
    t0 = time.time()

    for imgs, hard_masks, teacher_probs in train_loader:
        imgs          = imgs.to(DEVICE)
        hard_masks    = hard_masks.to(DEVICE).float()
        teacher_probs = teacher_probs.to(DEVICE).float()

        with torch.cuda.amp.autocast():
            out  = student(imgs)['out']
            out  = F.interpolate(out, size=hard_masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(out, hard_masks, teacher_probs)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        train_loss += loss.item()

    scheduler.step()

    tr_loss = train_loss / len(train_loader)
    vl_loss, vl_f1 = val_epoch(student, val_loader, criterion, DEVICE)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    val_f1s.append(vl_f1)

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        best_epoch  = epoch
        torch.save(student.state_dict(), str(STUDENT_MODEL_PATH))

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or epoch == HEAD_EPOCHS + 1:
        marker = ' <-- best' if epoch == best_epoch else ''
        print(f'[S2] Epoch {epoch:3d}/{NUM_EPOCHS} | '
              f'TrLoss: {tr_loss:.4f} | VlLoss: {vl_loss:.4f} | '
              f'VlF1: {vl_f1:.4f} | LR: {lr_now:.2e} | {elapsed:.1f}s{marker}')

print('-' * 60)
print(f'Training complete — Best val F1: {best_val_f1:.4f} @ epoch {best_epoch}')
print(f'Student model saved: {STUDENT_MODEL_PATH}')

## 10. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='steelblue')
axes[0].plot(val_losses,   label='Val Loss',   color='coral')
axes[0].axvline(best_epoch - 1, color='green',  linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[0].axvline(HEAD_EPOCHS - 1, color='purple', linestyle=':', alpha=0.7, label='Stage 1→2 boundary')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(val_f1s, label='Val F1', color='mediumseagreen')
axes[1].axvline(best_epoch - 1,   color='green',  linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[1].axvline(HEAD_EPOCHS - 1,  color='purple', linestyle=':', alpha=0.7, label='Stage 1→2 boundary')
axes[1].axhline(best_val_f1,      color='orange', linestyle=':', alpha=0.7, label=f'Best F1={best_val_f1:.4f}')
axes[1].axhline(0.70, color='red', linestyle='--', alpha=0.5, label='Target F1=0.70')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Score')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(
    'Phase 5 Training — Student Distillation (MobileNetV3) — Predial Vision MX',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(str(CURVES_PATH), dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved: {CURVES_PATH}')

## 11. TTA Functions

In [ ]:
@torch.no_grad()
def tta_predict_chip(model, chip, device):
    """chip: (1, C, H, W) tensor on device. Returns averaged probability (1, 1, H, W)."""
    preds = []
    with torch.cuda.amp.autocast():
        # Original
        preds.append(torch.sigmoid(model(chip)['out']))
        # Horizontal flip
        f = torch.flip(chip, [3])
        preds.append(torch.flip(torch.sigmoid(model(f)['out']), [3]))
        # Vertical flip
        f = torch.flip(chip, [2])
        preds.append(torch.flip(torch.sigmoid(model(f)['out']), [2]))
        # Rotate 90
        r = torch.rot90(chip, 1, [2, 3])
        preds.append(torch.rot90(torch.sigmoid(model(r)['out']), 3, [2, 3]))
        # Rotate 180
        r = torch.rot90(chip, 2, [2, 3])
        preds.append(torch.rot90(torch.sigmoid(model(r)['out']), 2, [2, 3]))
        # Rotate 270
        r = torch.rot90(chip, 3, [2, 3])
        preds.append(torch.rot90(torch.sigmoid(model(r)['out']), 1, [2, 3]))
    return torch.stack(preds).mean(0)


infer_norm = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


def tta_sliding_window(model, imagery, device, chip_size=256, stride=128):
    """Run TTA inference over full image. imagery: CxHxW uint8. Returns probability map HxW float32."""
    if imagery.ndim == 3 and imagery.shape[0] <= 4:
        img_hwc = np.transpose(imagery, (1, 2, 0))  # CxHxW -> HxWxC
    else:
        img_hwc = imagery

    if img_hwc.shape[2] == 1:
        img_hwc = np.repeat(img_hwc, 3, axis=2)
    elif img_hwc.shape[2] > 3:
        img_hwc = img_hwc[:, :, :3]

    H, W       = img_hwc.shape[:2]
    pred_mask  = np.zeros((H, W), dtype=np.float32)
    count_mask = np.zeros((H, W), dtype=np.float32)

    model.eval()
    total = 0
    with torch.no_grad():
        for y in range(0, H - chip_size + 1, stride):
            for x in range(0, W - chip_size + 1, stride):
                chip = img_hwc[y:y+chip_size, x:x+chip_size]
                if chip.mean() < 0.02 * 255:
                    continue
                aug    = infer_norm(image=chip)
                chip_t = aug['image'].unsqueeze(0).to(device)
                prob   = tta_predict_chip(model, chip_t, device)
                prob_np = prob[0, 0].cpu().float().numpy()
                pred_mask[y:y+chip_size,  x:x+chip_size]  += prob_np
                count_mask[y:y+chip_size, x:x+chip_size]  += 1
                total += 1
                if total % 500 == 0:
                    print(f'  {total} chips processed')

    count_mask[count_mask == 0] = 1
    return pred_mask / count_mask


print('TTA functions defined — 6 augmentations per chip (fp16)')

## 12. TTA Inference on Original Imagery

In [ ]:
# Load best student weights
print(f'Loading best student model from {STUDENT_MODEL_PATH}...')
student.load_state_dict(torch.load(str(STUDENT_MODEL_PATH), map_location=DEVICE))
student.eval()
print('Best student model loaded')

print('\n=== TTA inference (MobileNetV3 Student) ===')
imagery_chw  = np.transpose(IMG_ARRAY_EFF, (2, 0, 1))  # HxWxC -> CxHxW
student_prob = tta_sliding_window(student, imagery_chw, DEVICE, chip_size=CHIP_SIZE, stride=STRIDE_INFER)

print(f'\nStudent soft probability map:')
print(f'  Shape: {student_prob.shape}')
print(f'  Min:   {student_prob.min():.4f}')
print(f'  Max:   {student_prob.max():.4f}')
print(f'  Mean:  {student_prob.mean():.4f}')
print(f'  >0.5:  {(student_prob > 0.5).mean()*100:.2f}% pixels')

## 13. Threshold Sweep

In [ ]:
def evaluate_at_threshold(pred, target, threshold):
    pred_bin = (pred > threshold).astype(np.float32)
    target_f = target.astype(np.float32)
    tp   = float((pred_bin * target_f).sum())
    fp   = float((pred_bin * (1 - target_f)).sum())
    fn   = float(((1 - pred_bin) * target_f).sum())
    prec = tp / (tp + fp + 1e-7)
    rec  = tp / (tp + fn + 1e-7)
    f1   = 2 * prec * rec / (prec + rec + 1e-7)
    iou  = tp / (tp + fp + fn + 1e-7)
    return prec, rec, f1, iou

H_p, W_p   = student_prob.shape
MASK_ALIGNED = MASK_EFF[:H_p, :W_p]

sweep_results = {}
best_f1       = 0.0
best_thresh   = PRED_THRESHOLD

print('Threshold sweep — Student probability map vs MS Buildings mask:')
print(f'{"Threshold":>10} | {"Precision":>10} | {"Recall":>8} | {"F1":>8} | {"IoU":>8}')
print('-' * 56)

for t in THRESHOLDS:
    p, r, f1, iou = evaluate_at_threshold(student_prob, MASK_ALIGNED, t)
    sweep_results[t] = {'precision': p, 'recall': r, 'f1': f1, 'iou': iou}
    marker = ' <-- best' if f1 > best_f1 else ''
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = t
    print(f'{t:>10.2f} | {p:>10.4f} | {r:>8.4f} | {f1:>8.4f} | {iou:>8.4f}{marker}')

print(f'\nBest threshold: {best_thresh:.2f} (F1={best_f1:.4f})')

## 14. Geometric Filter and Vectorize

In [ ]:
def rectangularity(poly):
    mbr = poly.minimum_rotated_rectangle
    return poly.area / mbr.area if mbr.area > 0 else 0.0

def compactness(poly):
    p = poly.length
    return (4 * np.pi * poly.area / (p * p)) if p > 0 else 0.0

def filter_building(poly, min_area_m2=20, min_rect=0.45, min_compact=0.1):
    area_m2 = poly.area * (111000 ** 2)
    if area_m2 < min_area_m2:
        return False
    if rectangularity(poly) < min_rect:
        return False
    if compactness(poly) < min_compact:
        return False
    return True

def vectorize_mask(binary_mask, transform, crs):
    polys = []
    for geom, val in shapes(binary_mask.astype(np.uint8), transform=transform):
        if val == 1:
            poly = shape(geom)
            if poly.is_valid and not poly.is_empty:
                polys.append(poly)
    return polys

binary_pred = (student_prob > best_thresh).astype(np.uint8)
print(f'Vectorizing at threshold={best_thresh:.2f}...')
raw_polys = vectorize_mask(binary_pred, IMG_TRANSFORM, IMG_CRS)
print(f'Raw polygons before geometric filter: {len(raw_polys):,}')

before_count   = len(raw_polys)
filtered_polys = [p for p in raw_polys if filter_building(p, MIN_AREA_M2, MIN_RECT, MIN_COMPACT)]
after_count    = len(filtered_polys)
removed        = before_count - after_count

print(f'Before geometric filter: {before_count:,} polygons')
print(f'After geometric filter:  {after_count:,} polygons (removed {removed:,})')

## 15. Final Evaluation — Comprehensive Comparison Across All Phases

In [ ]:
# Re-rasterize filtered polygons for final pixel-level evaluation
print('Rasterizing filtered predictions for final evaluation...')
if filtered_polys:
    shapes_list = [(mapping(p), 1) for p in filtered_polys]
    final_pred_mask = rasterize(
        shapes_list,
        out_shape=(H_p, W_p),
        transform=IMG_TRANSFORM,
        fill=0, dtype=np.uint8, all_touched=True
    ).astype(np.float32)
else:
    final_pred_mask = np.zeros((H_p, W_p), dtype=np.float32)

final_p, final_r, final_f1, final_iou = evaluate_at_threshold(final_pred_mask, MASK_ALIGNED, 0.5)

print()
print('=' * 62)
print('PREDIAL VISION MX — FINAL RESULTS')
print('=' * 62)
print(f'{"":22s} {"v10(OSM)":>9s} {"v12(DiceBCE)":>12s} {"P1(TTA)":>8s} {"P3(Teacher)":>11s} {"P5(Distill)":>11s}')
print('-' * 62)
print(f'{"Best val F1:":22s} {"0.176":>9s} {"0.4935":>12s} {"0.5851":>8s} {"0.6476":>11s} {best_val_f1:>11.4f}')
print(f'{"Pixel F1 (sweep):": 22s} {"—":>9s} {"—":>12s} {"0.6041":>8s} {"0.6818":>11s} {best_f1:>11.4f}')
print(f'{"Precision:": 22s} {"34.7%":>9s} {"38%":>12s} {"46.3%":>8s} {"54.2%":>11s} {final_p*100:>10.1f}%')
print(f'{"Recall:": 22s} {"11.8%":>9s} {"70-91%":>12s} {"87%":>8s} {"92%":>11s} {final_r*100:>10.1f}%')
print(f'{"Polygons:": 22s} {"10":>9s} {"4,844":>12s} {"4,519":>8s} {"8,340":>11s} {after_count:>11,}')
print('=' * 62)
print()
print('PHASE 5 DETAIL:')
print(f'  Architecture:   DeepLabV3 MobileNetV3-Large (student)')
print(f'  Teacher:        DeepLabV3 ResNet101 (Phase 3)')
print(f'  Distill loss:   BCE({BCE_WEIGHT}) + Dice({DICE_WEIGHT}) + MSE({DISTILL_ALPHA})')
print(f'  Mixed prec:     fp16 (torch.cuda.amp)')
print(f'  Best epoch:     {best_epoch} / {NUM_EPOCHS}')
print(f'  Best val F1:    {best_val_f1:.4f}')
print(f'  Final F1:       {final_f1:.4f}  (Target ≥ 0.70: {"MET" if final_f1 >= 0.70 else "NOT MET"})')
print(f'  IoU:            {final_iou:.4f}')
print(f'  Polygons:       {after_count:,}')
print('=' * 62)

## 16. Save All Outputs

In [ ]:
# final_eval.json
eval_data = {
    'phase': 5,
    'model': 'DeepLabV3 MobileNetV3-Large (Student)',
    'teacher': 'DeepLabV3 ResNet101 (Phase 3)',
    'label_source': LABEL_SOURCE,
    'buildings_count': int(BUILDINGS_COUNT),
    'training': {
        'total_epochs':    NUM_EPOCHS,
        'head_epochs':     HEAD_EPOCHS,
        'head_lr':         HEAD_LR,
        'full_lr':         FULL_LR,
        'min_lr':          MIN_LR,
        'best_epoch':      int(best_epoch),
        'best_val_f1':     float(best_val_f1),
        'batch_size':      BATCH_SIZE,
        'chip_size':       CHIP_SIZE,
        'stride_train':    STRIDE_TRAIN,
        'mixed_precision': True,
        'tta_passes':      6
    },
    'loss_weights': {
        'bce':     BCE_WEIGHT,
        'dice':    DICE_WEIGHT,
        'distill': DISTILL_ALPHA,
        'pos_weight': POS_WEIGHT
    },
    'threshold_sweep': {
        str(t): {k: float(v) for k, v in v2.items()}
        for t, v2 in sweep_results.items()
    },
    'best_threshold':       float(best_thresh),
    'before_geom_filter':   int(before_count),
    'after_geom_filter':    int(after_count),
    'removed_by_filter':    int(removed),
    'final_metrics': {
        'precision': float(final_p),
        'recall':    float(final_r),
        'f1':        float(final_f1),
        'iou':       float(final_iou),
        'polygons':  int(after_count),
        'target_met': bool(final_f1 >= 0.70)
    },
    'geometry_filter': {
        'min_area_m2': MIN_AREA_M2,
        'min_rect':    MIN_RECT,
        'min_compact': MIN_COMPACT
    },
    'phase_comparison': {
        'v10_osm':       {'best_val_f1': 0.176,  'polygons': 10},
        'v12_dicebce':   {'best_val_f1': 0.4935, 'polygons': 4844},
        'p1_tta':        {'best_val_f1': 0.5851, 'pixel_f1': 0.6041, 'precision': 0.463, 'recall': 0.87,  'polygons': 4519},
        'p3_teacher':    {'best_val_f1': 0.6476, 'pixel_f1': 0.6818, 'precision': 0.542, 'recall': 0.92,  'polygons': 8340},
        'p5_distill':    {'best_val_f1': float(best_val_f1), 'pixel_f1': float(best_f1),
                          'precision': float(final_p), 'recall': float(final_r), 'polygons': int(after_count)}
    }
}

with open(str(EVAL_PATH), 'w') as f:
    json.dump(eval_data, f, indent=2)
print(f'final_eval.json saved: {EVAL_PATH}')

In [ ]:
# final_buildings.geojson
if filtered_polys:
    features = []
    for i, poly in enumerate(filtered_polys):
        area_m2 = poly.area * (111000 ** 2)
        features.append({
            'type': 'Feature',
            'geometry': mapping(poly),
            'properties': {
                'id':             i,
                'area_m2':        round(float(area_m2), 2),
                'rectangularity': round(float(rectangularity(poly)), 4),
                'compactness':    round(float(compactness(poly)), 4),
                'source':         'phase5_student_mobilenetv3'
            }
        })
    geojson_out = {
        'type': 'FeatureCollection',
        'crs': {'type': 'name', 'properties': {'name': str(IMG_CRS)}},
        'features': features
    }
    with open(str(GEOJSON_PATH), 'w') as f:
        json.dump(geojson_out, f)
    print(f'final_buildings.geojson saved: {GEOJSON_PATH} ({len(features):,} buildings)')
else:
    print('No polygons to save')

In [ ]:
# final_visualization.png
VIZ_CROP_SIZE = min(512, H_p, W_p)
cy = max(0, (H_p - VIZ_CROP_SIZE) // 2)
cx = max(0, (W_p - VIZ_CROP_SIZE) // 2)

img_crop    = IMG_ARRAY_EFF[cy:cy+VIZ_CROP_SIZE, cx:cx+VIZ_CROP_SIZE]
gt_crop     = MASK_ALIGNED[cy:cy+VIZ_CROP_SIZE,  cx:cx+VIZ_CROP_SIZE]
teach_crop  = teacher_prob[cy:cy+VIZ_CROP_SIZE,  cx:cx+VIZ_CROP_SIZE]
stud_crop   = student_prob[cy:cy+VIZ_CROP_SIZE,  cx:cx+VIZ_CROP_SIZE]
pred_crop   = binary_pred[cy:cy+VIZ_CROP_SIZE,   cx:cx+VIZ_CROP_SIZE]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))

axes[0].imshow(img_crop)
axes[0].set_title('RGB Imagery', fontsize=11)
axes[0].axis('off')

axes[1].imshow(img_crop)
axes[1].imshow(gt_crop, alpha=0.5, cmap='Reds', vmin=0, vmax=1)
axes[1].set_title(f'GT Mask\n({BUILDINGS_COUNT:,} buildings)', fontsize=11)
axes[1].axis('off')

im2 = axes[2].imshow(teach_crop, cmap='viridis', vmin=0, vmax=1)
axes[2].set_title('Teacher Soft Prob\n(ResNet101 + TTA)', fontsize=11)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

im3 = axes[3].imshow(stud_crop, cmap='plasma', vmin=0, vmax=1)
axes[3].set_title('Student Soft Prob\n(MobileNetV3 + TTA)', fontsize=11)
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

axes[4].imshow(img_crop)
axes[4].imshow(pred_crop, alpha=0.5, cmap='Blues', vmin=0, vmax=1)
axes[4].set_title(f'Student Prediction\nthr={best_thresh:.2f} | F1={final_f1:.4f}', fontsize=11)
axes[4].axis('off')

red_patch  = mpatches.Patch(color='red',    alpha=0.6, label='Ground Truth')
blue_patch = mpatches.Patch(color='blue',   alpha=0.6, label='Student Prediction')
fig.legend(handles=[red_patch, blue_patch], loc='lower center', ncol=2, fontsize=11)

plt.suptitle(
    f'Predial Vision MX — Phase 5: Knowledge Distillation (MobileNetV3 Student)\n'
    f'F1={final_f1:.4f} | P={final_p:.4f} | R={final_r:.4f} | Polygons={after_count:,} | '
    f'Target (≥0.70): {"MET" if final_f1 >= 0.70 else "NOT MET"}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(str(VIZ_PATH), dpi=150, bbox_inches='tight')
plt.show()
print(f'Visualization saved: {VIZ_PATH}')

In [ ]:
# Final output summary
print('\n' + '=' * 60)
print('=== PHASE 5 OUTPUT FILES ===')
print('=' * 60)
output_files = [
    STUDENT_MODEL_PATH, GEOJSON_PATH, EVAL_PATH, VIZ_PATH, CURVES_PATH
]
for p in output_files:
    exists = Path(p).exists()
    size   = Path(p).stat().st_size / 1e6 if exists else 0
    status = f'{size:.2f} MB' if exists else 'MISSING'
    print(f'  {Path(p).name:45s} {status}')
print('=' * 60)
print()
print('PHASE 5 COMPLETE.')
print(f'  Student model:  DeepLabV3 MobileNetV3-Large')
print(f'  Teacher model:  DeepLabV3 ResNet101 (Phase 3)')
print(f'  Best Val F1:    {best_val_f1:.4f}')
print(f'  Final F1:       {final_f1:.4f}  (Target ≥ 0.70: {"MET" if final_f1 >= 0.70 else "NOT MET"})')
print(f'  Polygons:       {after_count:,}')
print()
print('PROJECT COMPLETE — Predial Vision MX pipeline (Phases 1-5)')